# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook guides you through loading, inspecting, and processing the FAIR^2 dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.
It demonstrates how to reference all entities by their Croissant `@id` attributes for transparent and reproducible data processing.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant (run only if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL provided:
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Initialize the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)
# Fetch and print high-level metadata
meta = dataset.metadata

print(f"Dataset title: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Published: {getattr(meta, 'datePublished', 'unknown')}")
print(f"License: {getattr(meta, 'license', 'unknown')}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), their fields (`cr:Field`) and columns, including the `@id` for each.

We identify the data structure, so all analysis fields are referenced using their Croissant `@id`.

In [ ]:
# List all record sets by @id, name, and their field @ids
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
overview = {}
for rset in record_sets:
    print(f"RecordSet: {rset.id}")
    print(f"  Name: {getattr(rset, 'name', '(no name)')}")
    # List all fields for this record set, by @id, name, and datatype
    fields_list = []
    for field in getattr(rset, 'fields', []):
        print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', '')} | DataType: {getattr(field, 'data_type', '')}")
        fields_list.append(field.id)
    print()
    overview[rset.id] = fields_list

## 3. Data Extraction
Load data from a specific record set into a `pandas` DataFrame using record set and field `@id` values from above.

**Note:** All data reference is performed by using Croissant `@id` for both record sets and fields (columns).

In [ ]:
# We'll load the first record set as an example (you can modify or extend as needed)
dfs = {}
for rset in record_sets:
    records = list(dataset.records(record_set=rset.id))
    dfs[rset.id] = pd.DataFrame(records)

# Show columns for each extracted DataFrame
for set_id, df in dfs.items():
    print(f"DataFrame for RecordSet '{set_id}': Columns: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from a record set, filter, normalize, and group.

You can replace the example field `@id`'s below with those relevant for your use case, referencing from the above data overview.

In [ ]:
import numpy as np
# Choose the record set and field by @id (Customise these based on your dataset overview above):

# Example (please adapt accordingly after viewing section 2 output!):
# record_set_id = 'cr:RecordSet_Proteins'         # Replace with the actual @id
# numeric_field_id = 'cr:Field_MolWeight'        # Replace with the actual @id (e.g. molecular weight)
# group_field_id = 'cr:Field_Sample'             # Replace with the @id of a categorical/sample/group field

# For illustration, auto-assigns the first available numeric-looking field
record_set_id = next(iter(dfs))  # e.g. the first record set
numeric_field_id = None
df = dfs[record_set_id]
for col in df.columns:
    # Heuristically treat fields with float or int values as numeric fields (change as needed)
    if df[col].dtype in [np.float64, np.int64] or np.issubdtype(df[col].dtype, np.number):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to find a likely numeric field by content sample
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col], errors='raise')
            numeric_field_id = col
            df[col] = pd.to_numeric(df[col], errors='coerce')
            break
        except Exception:
            continue

if numeric_field_id is not None:
    print(f\
        f"Using record set: {record_set_id}\nUsing numeric field: {numeric_field_id}\n")
    # Filter records with value above a threshold
    threshold = np.nanquantile(df[numeric_field_id], 0.5)  # median as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (median): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[numeric_field_id + "_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized field '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Try grouping by a categorical field (if any string field exists)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        grouped = (
            filtered_df.groupby(group_field_id)[numeric_field_id]
            .mean().reset_index()
        )
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped.head())
else:
    print("No numeric field found in this record set.")

## 5. Visualization
We visualize the distribution of the selected numeric field, and, if grouped, the mean statistic per group.

Replace fields and record sets with your Croissant `@id` values for custom visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not df[numeric_field_id].isna().all():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    if 'grouped' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading a Croissant-annotated FAIR^2 dataset via `mlcroissant`, listing all record sets and fields by `@id`, and referencing all analysis steps by these stable identifiers.

You can adapt this notebook to process any Croissant dataset by referencing entities through their `@id` in every cell.